In [69]:
import pandas as pd
import numpy as np

season = "2023q2"
fac1 = pd.read_feather(rf"/home/user96/Factor_splitting/fac20250613_de_score_{season}.fea")

fac2 = np.loadtxt(rf"/home/user96/20250903_trainmodel2/model_test/nn--fac20250613_ic1500--label1--dropout0.2/ALL{season}-feature_map.fea", dtype=str) 

In [70]:
topindex_list = fac1.nlargest(1500, 'value')['index'].tolist()
topindex_list.sort()


In [71]:
fac2_list = [ name.split('=')[0] for name in fac2.tolist()]
fac2_list.sort()


In [72]:
# 可将下面两个示例列表替换为你的 A 与 B（可以包含重复元素）。
A = topindex_list
B = fac2_list

from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

def overlap_metrics(A, B, treat_duplicates="set"):
    """
    计算 A 与 B 的重合程度。
    treat_duplicates:
      - "set": 将列表当作集合（忽略重复）
      - "multiset": 将列表当作多重集（保留重复）
    返回: metrics(dict), parts(dict)
    """
    if treat_duplicates not in ("set", "multiset"):
        raise ValueError("treat_duplicates 需为 'set' 或 'multiset'")
    
    if treat_duplicates == "set":
        SA, SB = set(A), set(B)
        inter = SA & SB
        union = SA | SB
        a_only = SA - SB
        b_only = SB - SA
        lenA, lenB = len(SA), len(SB)
        inter_n, union_n = len(inter), len(union)
        a_only_n, b_only_n = len(a_only), len(b_only)
    else:
        cA, cB = Counter(A), Counter(B)
        keys = set(cA) | set(cB)
        inter_n = sum(min(cA[k], cB[k]) for k in keys)
        union_n = sum(max(cA[k], cB[k]) for k in keys)
        lenA, lenB = sum(cA.values()), sum(cB.values())
        a_only_n = sum(max(cA[k] - cB[k], 0) for k in keys)
        b_only_n = sum(max(cB[k] - cA[k], 0) for k in keys)
    
    # 常用相似度 / 重合度指标
    jaccard = inter_n / union_n if union_n else 0.0
    dice = (2 * inter_n) / (lenA + lenB) if (lenA + lenB) else 0.0
    overlap_coeff = inter_n / min(lenA, lenB) if min(lenA, lenB) else 0.0
    precision_A_as_pred = inter_n / lenA if lenA else 0.0  # A当作"预测"
    recall_A_as_pred = inter_n / lenB if lenB else 0.0     # B当作"真值"
    
    metrics = {
        "模式": treat_duplicates,
        "|A|": lenA,
        "|B|": lenB,
        "|A∩B|": inter_n,
        "|A∪B|": union_n,
        "A仅有": a_only_n,
        "B仅有": b_only_n,
        "Jaccard相似度": jaccard,
        "Dice系数": dice,
        "Overlap系数": overlap_coeff,
        "Precision(A→B)": precision_A_as_pred,
        "Recall(A→B)": recall_A_as_pred,
        "F1(A→B)": (2 * precision_A_as_pred * recall_A_as_pred / (precision_A_as_pred + recall_A_as_pred) 
                    if (precision_A_as_pred + recall_A_as_pred) else 0.0),
    }
    parts = {
        "A_only": a_only_n,
        "Both": inter_n,
        "B_only": b_only_n,
    }
    return metrics, parts

# 计算两种模式下的指标
metrics_set, parts_set = overlap_metrics(A, B, "set")
metrics_set


{'模式': 'set',
 '|A|': 1500,
 '|B|': 1500,
 '|A∩B|': 747,
 '|A∪B|': 2253,
 'A仅有': 753,
 'B仅有': 753,
 'Jaccard相似度': 0.33155792276964047,
 'Dice系数': 0.498,
 'Overlap系数': 0.498,
 'Precision(A→B)': 0.498,
 'Recall(A→B)': 0.498,
 'F1(A→B)': 0.498}